# DeepLOB Baseline Notebook
This notebook implements and trains the baseline encoder model under our unified, leakage-safe LOBench replication pipeline.

In [1]:
# Mount Google Drive if running in Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.chdir('/content/drive/MyDrive/JEPA_LOB/baselines')
    print('Mounted Google Drive and changed directory to baselines.')
except ImportError:
    print('Running locally or Google Drive mount skipped.')

Mounted at /content/drive
Mounted Google Drive and changed directory to baselines.


In [2]:
# Install PyTorch Lightning if it is not present in the environment
!pip install -q lightning pandas numpy torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 42.3 MB/s eta 0:00:00


In [3]:
from common import *
import torch
import torch.nn as nn
import torch.nn.functional as F

print('Libraries and common module imported successfully.')

Libraries and common module imported successfully.


In [4]:
class DeepLOBEncoder(nn.Module):
    def __init__(self, latent_dim=256):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=(1,2), stride=(1,2)),
            nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, kernel_size=(4,1)),
            nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, kernel_size=(4,1)),
            nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 32, kernel_size=(1,2), stride=(1,2)),
            nn.Tanh(), nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, kernel_size=(4,1)),
            nn.Tanh(), nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, kernel_size=(4,1)),
            nn.Tanh(), nn.BatchNorm2d(32),
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 32, kernel_size=(1,10)),
            nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, kernel_size=(4,1)),
            nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, kernel_size=(4,1)),
            nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
        )
        self.inp1 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=(1,1), padding='same'),
            nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
            nn.Conv2d(64, 64, kernel_size=(3,1), padding='same'),
            nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
        )
        self.inp2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=(1,1), padding='same'),
            nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
            nn.Conv2d(64, 64, kernel_size=(5,1), padding='same'),
            nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
        )
        self.inp3 = nn.Sequential(
            nn.MaxPool2d((3,1), stride=(1,1), padding=(1,0)),
            nn.Conv2d(32, 64, kernel_size=(1,1), padding='same'),
            nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
        )
        self.lstm = nn.LSTM(input_size=192, hidden_size=64, num_layers=1, batch_first=True)
        self.proj = nn.Linear(64, latent_dim)

    def forward(self, x):  # x: [B, 100, 40]
        x = x.unsqueeze(1)                       # [B, 1, 100, 40]
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x_inp1, x_inp2, x_inp3 = self.inp1(x), self.inp2(x), self.inp3(x)
        x = torch.cat((x_inp1, x_inp2, x_inp3), dim=1)
        x = x.permute(0, 2, 1, 3)
        x = torch.reshape(x, (-1, x.shape[1], x.shape[2]))
        x, _ = self.lstm(x)
        last = x[:, -1, :]
        return self.proj(last)

In [ ]:
model_name = 'DeepLOB'
stocks = ['sz000001', 'sz000002', 'sz000858', 'sz300147', 'sz002415']
for stock in stocks:
    print(f'\n========================================')
    print(f'Starting experiment for Model: {model_name} | Stock: {stock}')
    print(f'========================================')
    run_experiment(
        encoder_class=DeepLOBEncoder,
        model_name=model_name,
        stock=stock,
        latent_dim=256,
        max_epochs=100
    )

Testing ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450/450 0:00:08 • 0:00:00 51.52it/s

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/.shortcut-targets-by-id/1CU14Fp2kwwtsMDvchQKKm-sSCYR6c_rS/JEPA_LOB/baselines/training_logs/DeepLOB/sz000858/metrics.csv'